In [14]:
# --- STEP 1: READ AND CHECK OUR FILE.TXT ---
file_name = 'IPcheckAlienVault.txt'

with open(file_name, 'r', encoding='utf-8') as f:
    my_ips = [line.strip() for line in f if line.strip()]

print(f"Total IPs found: {len(my_ips)}")
print("-" * 30)

# 1. Show the Top 3
print("TOP 3 IPs:")
for ip in my_ips[:3]:
    print(f"  > {ip}")

print("-" * 30)

# 2. Show the Bottom 3
print("BOTTOM 3 IPs:")
for ip in my_ips[-3:]:
    print(f"  > {ip}")

Total IPs found: 5
------------------------------
TOP 3 IPs:
  > 34.104.35.123
  > 216.180.246.139
  > 115.164.119.170
------------------------------
BOTTOM 3 IPs:
  > 115.164.119.170
  > 115.164.63.132
  > 115.164.63.194


In [ ]:
import requests

# --- CHANGE 1: Use your Alienvault OTX Key ---
# Get it from: https://otx.alienvault.com/settings
API_KEY = #YOUR_API_KEY_HERE

def test_alienvault_api():
    # --- CHANGE 2: Updated URL for Alienvault IPv4 Indicator ---
    url = 'https://otx.alienvault.com/api/v1/indicators/IPv4/8.8.8.8/general'
    
    # --- CHANGE 3: Header must be 'X-OTX-API-KEY' ---
    headers = {
        'X-OTX-API-KEY': API_KEY,
        'Accept': 'application/json'
    }
    
    try:
        response = requests.get(url, headers=headers)
        
        if response.status_code == 200:
            print("✅ Alienvault OTX Success!")
            return True
        else:
            print(f"❌ Failed: {response.status_code} - {response.text}")
            return False
    except Exception as e:
        print(f"❌ Error: {e}")
        return False

api_works = test_alienvault_api()

✅ Alienvault OTX Success!


In [ ]:
import requests
import pprint

# --- CONFIGURATION ---
# Replace with your actual Alienvault OTX Key
API_KEY = #YOUR_API_KEY 
IP_TO_CHECK = '8.8.8.8'

# 1. Alienvault OTX General Indicator endpoint
url = f'https://otx.alienvault.com/api/v1/indicators/IPv4/{IP_TO_CHECK}/general'

# 2. Alienvault uses "X-OTX-API-KEY" header
headers = {
    'Accept': 'application/json',
    'X-OTX-API-KEY': API_KEY
}

# 3. Request
response = requests.get(url, headers=headers)

# 4. Print results
if response.status_code == 200:
    print(f"--- RAW ALIENVAULT DATA FOR {IP_TO_CHECK} ---")
    pprint.pprint(response.json())
else:
    print(f"Error {response.status_code}: {response.text}")

--- RAW ALIENVAULT DATA FOR 8.8.8.8 ---
{'accuracy_radius': 1000,
 'area_code': 0,
 'asn': 'AS15169 google llc',
 'base_indicator': {'access_reason': '',
                    'access_type': 'public',
                    'content': '',
                    'description': '',
                    'id': 11911,
                    'indicator': '8.8.8.8',
                    'title': '',
                    'type': 'IPv4'},
 'charset': 0,
 'city': None,
 'city_data': True,
 'continent_code': 'NA',
 'country_code': 'US',
 'country_code2': 'US',
 'country_code3': 'USA',
 'country_name': 'United States of America',
 'dma_code': 0,
 'false_positive': [{'assessment': 'accepted',
                     'assessment_date': '2021-05-19T15:36:44.966000',
                     'report_date': '2021-03-16T14:46:19.003000'}],
 'flag_title': 'United States of America',
 'flag_url': '/assets/images/flags/us.png',
 'indicator': '8.8.8.8',
 'latitude': 37.751,
 'longitude': -97.822,
 'postal_code': None,
 'pulse_i

In [17]:
import sys
MASTER_DATA = []

if api_works:
    # Check the first 900 for now
    test_list = my_ips[:5] 
    
    for index, ip in enumerate(test_list):
        # Progress Counter
        sys.stdout.write(f"\r🔍 Checking: {index + 1} of {len(test_list)}... ({ip})")
        sys.stdout.flush()

        url = f'https://api.ipinfo.io/lite/{ip}'
        headers = {'Authorization': f'Bearer {API_KEY}'}
        
        try:
            response = requests.get(url, headers=headers)
            # The Lite JSON has: 'ip', 'as_name', 'country',
            MASTER_DATA.append(response.json())
        except:
            continue

🔍 Checking: 5 of 5... (115.164.63.194))

In [18]:
import pandas as pd

# 1. Check if we have data
if not MASTER_DATA:
    print("No data found in MASTER_DATA. Please run the Alienvault check loop first!")
else:
    # 2. Convert to Table
    df = pd.DataFrame(MASTER_DATA)

    # ALIENVAULT SPECIFIC COLUMNS
    # Based on your raw output, we map the keys to your desired headers
    columns_to_keep = {
        'indicator': 'IP Address',         # Found at the root of your JSON
        'country_name': 'Country Name',   # Found as 'United States of America'
        'continent_code': 'Continent',    # Found as 'NA'
        'asn': 'Organization/ISP',        # Found as 'AS15169 google llc'
        'reputation': 'Reputation Score', # Found as 0
        'pulse_count': 'Threat Pulses'
    }
    
    # Filter only what exists in the data (handling missing keys safely)
    existing = [c for c in columns_to_keep.keys() if c in df.columns]
    df_final = df[existing].rename(columns=columns_to_keep)

    # 3. Save to Excel
    file_name = "Alienvault_Report.xlsx"
    df_final.to_excel(file_name, index=False)
    
    print(f"\n✅ Excel Created: {file_name}")


✅ Excel Created: Alienvault_Report.xlsx
